# **Import Libraries**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, models, transforms
from PIL import Image
import pandas as pd
import numpy as np
from collections import Counter
import os
from torchvision.models import ResNet50_Weights, efficientnet_b3, EfficientNet_B3_Weights

# **Data Preprocessing**

In [ ]:
train_dir = '/kaggle/input/AI-OF-GOD-4/aog_data/train'
test_dir = '/kaggle/input/AI-OF-GOD-4/aog_data/test/images'
submission_dir = '/kaggle/input/AI-OF-GOD-4/aog_data/sample_submission.csv'

In [ ]:
batch_size = 128
epochs = 15
learning_rate = 0.001
val_split = 0.2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.ToTensor()
])

val_test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.ToTensor()
])

**Countering Class Imbalance**

In [ ]:
full_train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)

class_counts = Counter(full_train_dataset.targets)
total_samples = len(full_train_dataset)
num_classes = len(class_counts)

class_weights = {i: total_samples / (num_classes * class_counts[i]) for i in range(num_classes)}
class_weights_tensor = torch.tensor([class_weights[i] for i in range(num_classes)], dtype=torch.float).to(DEVICE)

In [ ]:
num_train_samples = len(full_train_dataset)
num_val_samples = int(num_train_samples * VALIDATION_SPLIT)
num_train_samples -= num_val_samples
train_subset, val_subset = random_split(full_train_dataset, [num_train_samples, num_val_samples])

In [ ]:
class ValDatasetWrapper(Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        original_pil_image = transforms.ToPILImage()( (img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)) + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1) )
        return self.transform(original_pil_image), label
val_dataset = ValDatasetWrapper(val_subset, val_test_transforms)

In [ ]:
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(root_dir) if os.path.isfile(os.path.join(root_dir, f))])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_files[idx]

test_dataset = TestDataset(TEST_DIR, transform=val_test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# **Model Initilization**

In [ ]:
def get_model(num_classes):
    model = models.efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    for block in model.features[-3:]:
        for param in block.parameters():
            param.requires_grad = True
    num_ftrs = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.BatchNorm1d(512),
        nn.Linear(512, num_classes)
    )
    return model

model = get_model(num_classes).to(DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam([
    {'params': model.classifier.parameters(), 'lr': LEARNING_RATE},
    {'params': model.features[-1].parameters(), 'lr': LEARNING_RATE / 10},
    {'params': model.features[-2].parameters(), 'lr': LEARNING_RATE / 50},
    {'params': model.features[-3].parameters(), 'lr': LEARNING_RATE / 100}
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# **Model Training**

In [ ]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    model.eval()
    val_running_loss = 0.0
    corrects = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_running_loss += loss.item() * inputs.size(0)
            corrects += torch.sum(preds == labels.data)
    val_loss = val_running_loss / len(val_loader.dataset)
    val_acc = corrects.double() / len(val_loader.dataset)
    scheduler.step()
    print(f'Epoch {epoch+1}/{epochs} -> Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}')

# **Submission and Prediction**

In [ ]:
model.eval()
predictions = []
filenames = []
with torch.no_grad():
    for inputs, fnames in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        predictions.extend(preds.cpu().numpy())
        filenames.extend(fnames)
submission_df = pd.DataFrame({
    'filename': filenames,
    'label': predictions
})
submission_df.to_csv(SUBMISSION_CSV_PATH, index=False)

In [ ]:
submission_df.to_csv("submission.csv", index=False)